# 4th Down Decision Model — Data Pull

Pulls 10 seasons (2016–2025, regular season + playoffs) of play-by-play data via `nflreadpy`, filters down to 4th-down plays, and builds a clean `decision` label (`go` / `punt` / `field_goal`) for each one.

In [ ]:
import sys
!{sys.executable} -m pip install pyarrow

In [ ]:
import pandas as pd
import nflreadpy as nfl

## Config

In [ ]:
SEASONS = list(range(2016, 2026))  # last 10 completed seasons

# Plays that don't represent a real 4th-down decision: penalties/pre-snap
# dead balls (no_play), missing play type, and kneel-outs to run clock.
EXCLUDED_PLAY_TYPES = {"no_play", "qb_kneel", "qb_spike"}

DECISION_MAP = {
    "punt": "punt",
    "field_goal": "field_goal",
    "pass": "go",
    "run": "go",
}

KEEP_COLUMNS = [
    "game_id",
    "season",
    "season_type",
    "week",
    "posteam",
    "defteam",
    "qtr",
    "down",
    "ydstogo",
    "yardline_100",
    "game_seconds_remaining",
    "half_seconds_remaining",
    "score_differential",
    "play_type",
    "decision",
    "field_goal_result",
    "epa",
    "wp",
    "wpa",
    "desc",
]

# "decision" doesn't exist yet at raw-pull time, it gets computed below
RAW_COLUMNS = [c for c in KEEP_COLUMNS if c != "decision"]

## Pull play-by-play and filter to 4th downs

`nflreadpy` returns a polars DataFrame natively. Converting the *entire* raw table (10 seasons x 372 columns, ~484k rows) to pandas in one shot uses a lot of memory (pandas is heavier than polars for wide, string-heavy tables) and can crash the kernel on memory-constrained machines like Codespaces.

Instead, we load and filter one season at a time: filter to 4th downs and select only the columns we need *while still in polars*, then convert just that small slice to pandas. Peak memory only ever holds one season's raw data, not all ten.

In [ ]:
print(f"Pulling play-by-play for seasons {SEASONS[0]}-{SEASONS[-1]}...")

season_frames = []
for season in SEASONS:
    raw = nfl.load_pbp([season])          # polars, one season at a time
    raw = raw.filter(raw["down"] == 4)    # cheap columnar filter before converting
    raw = raw[RAW_COLUMNS]                # drop the other ~350 columns we don't need
    season_frames.append(raw.to_pandas())
    print(f"  {season}: {raw.shape[0]:,} 4th-down plays")

pbp = pd.concat(season_frames, ignore_index=True)
pbp.shape

In [ ]:
fourth = pbp[~pbp["play_type"].isin(EXCLUDED_PLAY_TYPES)].copy()
fourth = fourth[fourth["play_type"].notna()]

fourth["decision"] = fourth["play_type"].map(DECISION_MAP)
fourth = fourth[fourth["decision"].isin(["punt", "field_goal", "go"])]

fourth = fourth[KEEP_COLUMNS]
fourth = fourth.dropna(
    subset=["ydstogo", "yardline_100", "score_differential", "game_seconds_remaining"]
)
fourth = fourth.reset_index(drop=True)

print(f"4th-down plays kept: {fourth.shape[0]:,} rows, {fourth.shape[1]} columns")

## Sanity check

In [ ]:
fourth["decision"].value_counts()

In [ ]:
fourth.head(10)

## Save

In [ ]:
out_parquet = "data/fourth_downs.parquet"
out_csv = "data/fourth_downs.csv"
fourth.to_parquet(out_parquet, index=False)
fourth.to_csv(out_csv, index=False)
print(f"Saved to {out_parquet} and {out_csv}")